## Import Libraries


In [ ]:
# Import Polars for dataframe analytics
import polars as pl
# Improt Path to manage file paths
from pathlib import Path

## Define Data Path


In [ ]:
# Define data folder path
data_folder = Path("../data")

## Load Cleaned Dataset


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

## Create Customer-Level Metrics


In [ ]:
# Aggregate customer level behavior
customer_metrics = (sales_df.group_by("customer_id").agg(

    # Total orders per customer
    pl.len().alias("total_orders"),

    # Total money spent
    pl.col("revenue").sum().alias("total_spent"),

    # Average order value
    pl.col("revenue").mean().alias("avg_order_value"),

    # First Purchase date
    pl.col("purchase_date").min().alias("first_purchase"),

    # Last purchase date
    pl.col("purchase_date").max().alias("last_purchase")

)
)
customer_metrics.head()

## Calculate Customer Lifetime


In [ ]:
# Calculate customer lifetime in days
customer_metrics = customer_metrics.with_columns(
    (pl.col("last_purchase") - pl.col("first_purchase")
     ).dt.total_days().alias("customer_lifetime_days")
)

customer_metrics.tail(10)

## Calculate Purchase Frequency


In [ ]:
# Purchase frequecy = orders / lifetime
customer_metrics = customer_metrics.with_columns(
    (
        pl.col("total_orders") / (pl.col("customer_lifetime_days")+1
                                  )
    ).alias("purchase_frequency")
)

# Note:- adding +1 avoids divide by 0 for single purchase customer
customer_metrics.head()

# Save Customer Level Dataset

This dataset will later power segmentation and churn analysis


In [ ]:
# Save customer level metrics
customer_metrics.write_csv(data_folder / "customer_level_metrics.csv")

# Save customer level metrics in parquet
customer_metrics.write_parquet(data_folder / "customer_level_metrics.parquet")

## Calculate Global KPIs

Computing business-level metrics.


In [ ]:
# Total customers
total_customers = sales_df.select(
    pl.col("customer_id").n_unique()
).item()

# Total orders
total_orders = sales_df.height

# Total revenue
total_revenue = sales_df.select(
    pl.col("revenue").sum()
).item()

## Calculate Ecommerce Metrics


In [ ]:
# Average order value
aov = total_revenue / total_orders

# Purhcase frequency
purchase_frequency = total_orders / total_customers

# Revenue per customer
revenue_per_customer = total_revenue / total_customers

## Repeat Purchase Rate


In [ ]:
# Customers with multiple purchases
repeat_customers = customer_metrics.filter(
    pl.col("total_orders") > 1
).height

repeat_purchase_rate = repeat_customers / total_customers

print("Repeat Customer:", repeat_customers)
print("Repeat Purchase Rate:", repeat_purchase_rate)

## Create KPI Tables


In [ ]:
# Create KPI dataframe
metrics_df = pl.DataFrame(
    {
        "metric": [
            "Total Customers",
            "Total Orders",
            "Total Revenue",
            "Average Order Value",
            "Purchase Frequency",
            "Revenue per Customer",
            "Repeat Customers",
            "Repeat Purchase Rate"
        ],
        "value": [
            float(total_customers),
            float(total_orders),
            float(total_revenue),
            round(aov, 2),
            round(purchase_frequency, 2),
            round(revenue_per_customer, 2),
            round(repeat_customers, 2),
            round(repeat_purchase_rate, 2)
        ]
    }
)

metrics_df

In [ ]:
metrics_df.sort("value", descending=True)

In [ ]:
customer_metrics.sort("total_orders", descending=True).head()

In [ ]:
customer_metrics.describe()

In [ ]:
customer_metrics.select([
    pl.col("total_orders").min().alias("min_orders"),
    pl.col("total_orders").mean().alias("avg_orders"),
    pl.col("total_orders").max().alias("max_orders")
])

## Save Metrics Dataset


In [ ]:
# Save KPI Dataset
metrics_df.write_csv(data_folder / "customer_behavior_metrics.csv")

metrics_df.write_parquet(data_folder / "customer_behavior_metrics.parquet")

# Adding Customer Segmentation

In [ ]:
customer_metrics = customer_metrics.with_columns(

    pl.when(pl.col("total_spent") > 30000).then(pl.lit("VIP"))
    .when(pl.col("total_orders") >= 30).then(pl.lit("Loyal"))
    .when(pl.col("total_orders") >= 10).then(pl.lit("Regular"))
    .otherwise(pl.lit("Low Value"))
    .alias("customer_type")

)

In [ ]:
customer_metrics.group_by("customer_type").count()

In [ ]:
sales_encriched = sales_df.join(
    customer_metrics.select(["customer_id", "customer_type"]),
    on="customer_id",
    how="left"
)

In [ ]:
sales_encriched.select("customer_type").unique()

In [ ]:
sales_encriched.write_parquet(data_folder / "sales_encriched.parquet")